# 14 - Gold 3: fact_status_transitions and fact_outage_interval

| Table | Grain (business key) |
|---|---|
| `fact_status_transitions` | one row per status change: (`measurement_id`, `source_sequence`) |
| `fact_outage_interval` | one row per outage interval: `outage_id` (tower + start time) |

Both load by MERGE, so a replay adds nothing.

In [0]:
BRONZE = "capstone6_telecom.bronze"
SILVER = "capstone6_telecom.silver"
GOLD   = "capstone6_telecom.gold"
cat    = GOLD   # this notebook writes to the gold schema

## 1. fact_status_transitions

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW _trans_src AS
SELECT h.measurement_id, h.source_sequence, h.change_ts, CAST(h.change_ts AS DATE) AS change_date,
       CAST(date_format(h.change_ts, 'yyyyMMdd') AS INT) AS date_key,
       r.region_key, t.tower_key, c.region_id, c.tower_id,
       h.previous_status, h.new_status, h.status_changed, h.change_reason,
       (h.source_sequence = c.last_sequence) AS is_latest_change,
       current_timestamp() AS updated_at
FROM {SILVER}.silver_status_history h
JOIN {SILVER}.silver_network_current c ON h.measurement_id = c.measurement_id
LEFT JOIN {cat}.dim_region r ON c.region_id = r.region_id
LEFT JOIN {cat}.dim_tower  t ON c.tower_id  = t.tower_id
""")

spark.sql(f"CREATE TABLE IF NOT EXISTS {cat}.fact_status_transitions AS SELECT * FROM _trans_src WHERE 1 = 0")

spark.sql(f"""
MERGE INTO {cat}.fact_status_transitions t
USING _trans_src s
ON t.measurement_id = s.measurement_id AND t.source_sequence = s.source_sequence
WHEN MATCHED AND NOT (t.is_latest_change <=> s.is_latest_change) THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

## 2. fact_outage_interval

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW _outage_src AS
SELECT o.outage_id, t.tower_key, o.tower_id, o.start_ts, CAST(o.start_ts AS DATE) AS start_date,
       CAST(date_format(o.start_ts, 'yyyyMMdd') AS INT) AS date_key,
       o.last_failed_ts, o.end_ts, o.is_open, o.outage_minutes, o.failed_readings,
       o.impacted_regions, o.impacted_region_count, o.affected_customers, o.alert_corroborated,
       current_timestamp() AS updated_at
FROM {SILVER}.silver_outage_intervals o
LEFT JOIN {cat}.dim_tower t ON o.tower_id = t.tower_id
""")

spark.sql(f"CREATE TABLE IF NOT EXISTS {cat}.fact_outage_interval AS SELECT * FROM _outage_src WHERE 1 = 0")

spark.sql(f"""
MERGE INTO {cat}.fact_outage_interval t
USING _outage_src s
ON t.outage_id = s.outage_id
WHEN MATCHED AND NOT (t.end_ts <=> s.end_ts AND t.outage_minutes <=> s.outage_minutes
                  AND t.failed_readings <=> s.failed_readings AND t.affected_customers <=> s.affected_customers
                  AND t.is_open <=> s.is_open)
  THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

## 3. Checks

In [0]:
display(spark.sql(f"""
SELECT (SELECT COUNT(*) FROM {cat}.fact_status_transitions) AS transitions,               -- expect 520
       (SELECT COUNT(*) - COUNT(DISTINCT measurement_id, source_sequence) FROM {cat}.fact_status_transitions) AS duplicate_keys,  -- 0
       (SELECT COUNT(*) FROM {cat}.fact_status_transitions WHERE is_latest_change) AS latest_changes,   -- 520 here (one change each)
       (SELECT COUNT(*) FROM {cat}.fact_outage_interval) AS outage_intervals,              -- about 2,108
       (SELECT COUNT(*) - COUNT(DISTINCT outage_id) FROM {cat}.fact_outage_interval) AS duplicate_outage_ids   -- 0"""))